# TartuNLP (entrenado en ruso) sobre el dev de SemEval-2027 – Subtask 2

Pipeline: **convertir** datos → **predecir** con el adaptador → **evaluar** F1.

Requisitos en este runtime de Colab:
- el entorno `/content/venv` del notebook de reproducción (si el runtime es nuevo, vuelve a ejecutar sus celdas de instalación),
- el adaptador entrenado (en Drive si usaste esa celda, o en `/content/ancient-lang-adapters/trained_models/ru`).

## 0. Traer nuestro repo al runtime
El kernel de Colab es una máquina remota: **no ve los archivos de tu ordenador**, aunque uses VS Code. Por eso clonamos el repo (haz `git push` antes).

In [5]:
REPO_URL = 'https://github.com/tonyOehme/hcc_practical_course_semeval2027.git'  # <-- cambiar (privado: https://<TOKEN>@github.com/...)
REPO_DIR = '/content/repo'
import os
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}/task2/semeval27_pipeline

/content/repo/task2/semeval27_pipeline


## 1. Drive: datos dev y adaptador
Sube la carpeta `dev/` (con `SV/` dentro) a `MyDrive/semeval27/dev`.

In [6]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/semeval27/dev'
ADAPTER = '/content/drive/MyDrive/axolotl_models/ru/classification'
# Si no guardaste el modelo en Drive:
# ADAPTER = '/content/ancient-lang-adapters/trained_models/ru/classification'
!ls {DATA_DIR} && ls {ADAPTER}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SV
adapter_config.json  pytorch_adapter.bin
head_config.json     pytorch_model_head.bin


In [4]:
!git -C /content/repo pull
!git -C /content/repo branch -a
!git -C /content/repo log --oneline -3
!find /content/repo -name "convert_data.py" -not -path "*/.git/*"
!ls /content/repo

Already up to date.
* main
  remotes/origin/HEAD -> origin/main
  remotes/origin/main
47748dc (HEAD -> main, origin/main, origin/HEAD) Task 2 trainiing with turtelXLM
aa08c88 first commit
/content/repo/task2/semeval27_pipeline/scripts/convert_data.py
README.md  task2


## 2. Convertir a pares (definición, uso)

In [7]:
!python scripts/convert_data.py --data_dir {DATA_DIR} --out_dir outputs/pairs

SV: 10 palabras, 1983 pares, 1983 etiquetados, 35.6% positivos → outputs/pairs/SV.pairs.tsv


## 3. Predecir con el adaptador de TartuNLP
Usa el Python del entorno de TartuNLP (`/content/venv`).

In [8]:
!/content/venv/bin/python scripts/predict_tartunlp.py \
    --pairs outputs/pairs/SV.pairs.tsv \
    --adapter_dir {ADAPTER} \
    --out outputs/preds/SV.tartunlp_ru.tsv

dispositivo: cuda
Some weights of XLMRobertaAdapterModel were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['heads.default.3.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100% 16/16 [00:17<00:00,  1.09s/it]
1983 pares puntuados → outputs/preds/SV.tartunlp_ru.tsv


## 4. Evaluar
Umbral 0.35 (el de TartuNLP) + barrido de umbrales. El mejor umbral del barrido está elegido sobre el propio dev: es optimista.

In [9]:
!python scripts/evaluate_subtask2.py --pred outputs/preds/SV.tartunlp_ru.tsv --threshold 0.35 --sweep

{
  "file": "SV.tartunlp_ru",
  "threshold": 0.35,
  "f1_macro_words": 0.4434,
  "f1_global": 0.4874,
  "n_usages": 1983,
  "n_words": 10,
  "baselines": {
    "always_1": {
      "f1_macro_words": 0.4717,
      "f1_global": 0.5246,
      "n_usages": 1983,
      "n_words": 10
    }
  }
}

Por palabra:
lang      word   n  pos_rate  pred_rate  precision  recall    f1
  SV     dumpa 145     0.310      0.469      0.250   0.378 0.301
  SV exportera 149     0.946      0.946      0.950   0.950 0.950
  SV       fru 274     0.894      0.748      0.878   0.735 0.800
  SV    fröken 174     0.057      0.868      0.060   0.900 0.112
  SV    förort 150     0.493      0.787      0.551   0.878 0.677
  SV     herre 283     0.071      1.000      0.071   1.000 0.132
  SV    kvinna 290     0.228      0.862      0.260   0.985 0.411
  SV     motiv 291     0.182      0.986      0.181   0.981 0.306
  SV      skär  98     0.276      0.867      0.282   0.889 0.429
  SV      suga 129     0.186      0.992      0.

## Análisis de resultados: TartuNLP (entrenado en ruso) → dev SV, Subtask 2

**Configuración:** adaptador de TartuNLP entrenado con AXOLOTL-24 ruso (5 épocas), aplicado sin reentrenar al dev sueco (10 palabras, 1983 usos, 35,6 % positivos). Transferencia cross-lingual zero-shot.

| | F1 macro por palabra | F1 global |
|---|---|---|
| TartuNLP-ru, umbral 0,35 | 0,443 | 0,487 |
| Mejor umbral del barrido (0,15)* | 0,472 | 0,525 |
| Baseline "siempre 1" | **0,472** | **0,525** |

\* Umbral elegido sobre el propio dev, así que el resultado es optimista.

**Conclusiones**
- El modelo **no supera la baseline trivial** con ningún umbral.
- **Predice positivo casi siempre:** pred_rate del 100 % en *herre*, 99 % en *suga* y *motiv*, cuando los positivos reales son el 7–19 %.
- **No discrimina:** la precisión coincide con la tasa de positivos en casi todas las palabras (*herre* 0,071 frente a 0,071, *motiv* 0,181 frente a 0,182), equivalente a elegir al azar.
- Los F1 altos de *exportera* (0,95) y *fru* (0,80) se deben a que el 95 % y el 89 % de sus usos son positivos, no a que el modelo acierte.

**Posibles causas**
1. Idioma distinto sin datos de entrenamiento en sueco.
2. Tarea distinta: en AXOLOTL se distinguen sentidos claramente diferentes; aquí las definiciones son matices sociales o de connotación (*förort* = barrio con mucha inmigración, *herre* = hombre de clase alta, *fröken* = mujer que ejerce una profesión).
3. Ruido de OCR en los textos de 1880–1915.
4. La entrada no marca la palabra objetivo.

**Siguientes pasos**
- Calcular el AUC por palabra para ver si las probabilidades tienen algo de señal.
- Reentrenar con datos suecos (validación cruzada dejando fuera una palabra cada vez) y marcar la palabra objetivo.
- Probar un modelo NLI multilingüe o un LLM.

In [10]:
import pandas as pd
from sklearn.metrics import roc_auc_score

df = pd.read_csv('outputs/preds/SV.tartunlp_ru.tsv', sep='\t')
auc = df.groupby('word').apply(lambda g: roc_auc_score(g['label'], g['prob'])).round(3)
print(auc.sort_values())
print('AUC medio:', auc.mean().round(3))
print(df.groupby('label')['prob'].describe().round(3))  # ¿difieren las probabilidades de positivos y negativos?

word
fru          0.381
dumpa        0.465
exportera    0.492
skär         0.524
fröken       0.562
suga         0.585
motiv        0.612
förort       0.635
kvinna       0.726
herre        0.847
dtype: float64
AUC medio: 0.583
        count   mean    std    min    25%    50%    75%    max
label                                                         
0      1278.0  0.539  0.143  0.083  0.435  0.552  0.651  0.822
1       705.0  0.509  0.146  0.091  0.407  0.509  0.620  0.835


/tmp/ipykernel_16200/3779780842.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  auc = df.groupby('word').apply(lambda g: roc_auc_score(g['label'], g['prob'])).round(3)


## Qué está pasando: ordenar bien no es lo mismo que acertar con el umbral

### 1. Qué produce el modelo
Para cada uso, el modelo da una **probabilidad** de que la definición describa ese uso. Después, el **umbral** convierte esa probabilidad en una decisión: si prob ≥ 0,35, se predice "sí" (1); si no, "no" (0).

Que el F1 salga bien depende de dos cosas distintas:
1. **Ordenar bien:** dentro de una palabra, ¿reciben los usos positivos probabilidades más altas que los negativos?
2. **Cortar bien:** ¿cae el umbral justo entre los dos grupos?

El F1 mezcla las dos. El **AUC** mide solo la primera.

### 2. Qué es el AUC
Coge al azar un uso positivo y uno negativo de la misma palabra. El AUC es la probabilidad de que el modelo le dé más puntuación al positivo.
- **1,0:** ordena perfectamente. Siempre existe un umbral que separa los dos grupos.
- **0,5:** ordena como una moneda al aire.
- **< 0,5:** ordena al revés.

El AUC no depende del umbral: dice si el modelo "sabe algo", aunque luego cortemos mal.

### 3. Lo que sale en nuestro dev
- **AUC medio: 0,583.** Hay algo de señal, pero poca.
- **Muy desigual entre palabras:**
  - señal clara: *herre* 0,85, *kvinna* 0,73;
  - señal débil: *förort*, *motiv*, *suga*, *fröken*, *skär* (0,52–0,64);
  - azar: *exportera* 0,49, *dumpa* 0,47;
  - al revés: *fru* 0,38.

### 4. La paradoja: cada palabra tiene su propio "nivel" de probabilidad
Si juntamos todos los usos, los **negativos tienen una probabilidad media más alta (0,539) que los positivos (0,509)**. Parecería que el modelo funciona al revés. Pero dentro de la mayoría de palabras ordena en la dirección correcta (AUC > 0,5).

Las dos cosas son compatibles porque **el nivel de probabilidad depende sobre todo de la palabra (de su definición), no del uso concreto**. Algunas definiciones parecen "compatibles" con casi cualquier frase y reciben probabilidades altas en todos sus usos. Otras reciben probabilidades bajas en todos.

**Ejemplo inventado para ilustrarlo:**

| | Negativos | Positivos | ¿Ordena bien? | Con umbral 0,5 |
|---|---|---|---|---|
| Palabra A | 0,60 – 0,70 | 0,75 – 0,80 | Sí (AUC = 1) | Todo "sí", y los negativos cuentan como fallos |
| Palabra B | 0,20 – 0,30 | 0,35 – 0,45 | Sí (AUC = 1) | Todo "no", y los positivos cuentan como fallos |

Las dos palabras están **perfectamente ordenadas**, pero ningún umbral común sirve para ambas. A A le haría falta uno de 0,72 y a B uno de 0,32.

### 5. El caso real más claro: *herre*
- AUC **0,85**: es la palabra que mejor ordena.
- F1 **0,13**: de las peores.
- El motivo es que todas sus probabilidades quedan por encima de 0,35. El modelo predice "sí" en el 100 % de los usos, cuando solo el 7 % son positivos. **La información está en el orden, pero el umbral único la desperdicia.**

### 6. Conclusión
El mal F1 tiene dos causas superpuestas:
1. **Poca señal en general** (AUC medio 0,58). En algunas palabras el modelo no sabe distinguir (*exportera*, *dumpa*) o se equivoca de dirección (*fru*).
2. **Mala calibración entre palabras.** Un umbral global no sirve porque cada palabra tiene su propia escala. Esto se puede atacar sin reentrenar, comparando cada uso con los demás usos **de su misma palabra** (normalización por palabra). Es lo siguiente que probamos.

In [11]:
import pandas as pd
from semeval27.metrics import summary

df = pd.read_csv('outputs/preds/SV.tartunlp_ru.tsv', sep='\t')
df['z'] = df.groupby('word')['prob'].transform(lambda x: (x - x.mean()) / x.std())

rows = []
for t in [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]:
    rows.append({'umbral_z': t, **summary(df.assign(pred=(df['z'] >= t).astype(int)))})
print(pd.DataFrame(rows).to_string(index=False))
print('Baseline siempre 1: 0.4717 (macro) / 0.5246 (global)')

 umbral_z  f1_macro_words  f1_global  n_usages  n_words
    -1.00          0.4677     0.5201      1983       10
    -0.75          0.4580     0.5090      1983       10
    -0.50          0.4499     0.5017      1983       10
    -0.25          0.4323     0.4848      1983       10
     0.00          0.4134     0.4650      1983       10
     0.25          0.3783     0.4280      1983       10
     0.50          0.3419     0.3870      1983       10
     0.75          0.3115     0.3379      1983       10
     1.00          0.2636     0.2720      1983       10
Baseline siempre 1: 0.4717 (macro) / 0.5246 (global)


## Experimento: normalizar las probabilidades por palabra (z-score)

**Idea:** como cada palabra tiene su propia escala de probabilidades, comparamos cada uso solo con los usos de su misma palabra: z = (prob − media de la palabra) / desviación de la palabra. Después aplicamos un umbral sobre z.

**Resultado:** no supera la baseline con ningún umbral.

| Umbral z | F1 macro por palabra | F1 global |
|---|---|---|
| −1,00 (mejor) | 0,468 | 0,520 |
| 0,00 | 0,413 | 0,465 |
| 1,00 | 0,264 | 0,272 |
| Baseline "siempre 1" | **0,472** | **0,525** |

**Por qué no funciona**
- Con z, un umbral fijo marca como positivos **aproximadamente la misma proporción de usos en todas las palabras** (con z ≥ 0, alrededor de la mitad).
- Pero la proporción real de positivos varía muchísimo: 6 % en *fröken*, 7 % en *herre*, 89 % en *fru*, 95 % en *exportera*. Normalizar ayuda en las palabras con pocos positivos, pero hunde las que tienen muchos.
- **"Siempre 1" es una baseline fuerte para el F1 de la clase positiva:** tiene recall 1, y en las palabras con muchos positivos da F1 ≈ 0,95 sin hacer nada.

**Conclusión del diagnóstico**
- El problema de fondo no es el umbral, es que **el modelo apenas distingue** positivos de negativos en sueco (AUC medio 0,58).
- Ninguna forma de fijar el umbral, global o por palabra, puede compensar eso. Para mejorar hay que mejorar el modelo:
  1. **Reentrenar con datos suecos:** validación cruzada dejando una palabra fuera cada vez, y marcando la palabra objetivo.
  2. **Cambiar de enfoque:** un modelo NLI multilingüe o un LLM que entienda mejor definiciones con matices sociales.

## 5. Guardar resultados en el repo (opcional)

In [ ]:
# Fichero de envío para Codabench: <LANG>/subtask2.jsonl + submissions/st2_tartunlp.zip
%cd /content/repo/task2/semeval27_pipeline
!git pull -q
!python scripts/make_submission.py --pred SV=outputs/preds/SV.tartunlp_ru.tsv \
    --threshold 0.35 --data_dir {DATA_DIR} --out submissions/st2_tartunlp
!head -n 3 submissions/st2_tartunlp/SV/subtask2.jsonl
!cp submissions/st2_tartunlp.zip /content/drive/MyDrive/semeval27/  # para descargarlo desde Drive

In [ ]:
!git status --short results/
# !git add results/ && git commit -m 'Resultados TartuNLP-ru en dev SV' && git push